# Q/ROUTE: six locations, 25 simulated qubits

Experimental fixed-depot QAOA. One depot + five targets; 25 target/position variables.
Classical Dijkstra supplies the costs; QAOA optimizes visiting order. A row-preserving
XY mixer retains one assignment per target; column uniqueness is still validated.

Select **Runtime > Change runtime type > T4 GPU** if available. GPU quotas and availability
are not guaranteed. This is NVIDIA GPU simulation, not quantum hardware.

Upload the accompanying `q_route_gpu_source.zip` in the next cell. This bundle contains
the actual project Python source, so the notebook does not rely on an unpushed GitHub branch.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
uploaded = files.upload()  # Choose q_route_gpu_source.zip from output/colab/.
archive = next((name for name in uploaded if name.endswith('.zip')), None)
if archive is None:
    raise RuntimeError('Upload the supplied source ZIP.')
root = Path.cwd() / 'qroute_gpu_source'
with zipfile.ZipFile(archive) as bundle:
    for name in bundle.namelist():
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise RuntimeError('Unsafe archive path')
    bundle.extractall(root)
assert (root / 'ibm_qiskit/qiskit_fixed.py').is_file()
print('Source ready:', root)


## Install pinned GPU dependencies

Use a fresh Colab runtime. Do not install the CPU `qiskit-aer` package alongside
`qiskit-aer-gpu`. If Colab requests a session restart after installation, restart
and rerun the next cells (the uploaded files remain in the runtime filesystem).


In [ ]:
import subprocess, sys
from pathlib import Path
root = Path.cwd() / 'qroute_gpu_source'
subprocess.run(['nvidia-smi'], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'requirements-gpu.txt')])


## Verify GPU selection, construct the model, and run

The GPU device is requested explicitly. Missing GPU support raises an error;
there is no silent CPU fallback. Start with 8 COBYLA evaluations, one QAOA layer,
and 4,096 shots. Runtime and solution quality are not guaranteed.


In [ ]:
import sys
from pathlib import Path
root = Path.cwd() / 'qroute_gpu_source'
sys.path.insert(0, str(root))
from ibm_qiskit.qiskit_fixed import aer_backend, build_fixed_problem
from ibm_qiskit.grid import build_grid
backend, options = aer_backend('GPU')
print('Requested simulator:', options['device'])
route_map, grid = build_grid(size=40, scenario='six')
print('Variables:', build_fixed_problem(route_map).get_num_binary_vars())
print('Actual grid search seconds:', grid['search_seconds'])


In [ ]:
import subprocess, sys
from pathlib import Path
root = Path.cwd() / 'qroute_gpu_source'
subprocess.run([sys.executable, '-u', '-m', 'ibm_qiskit.qiskit_experimental',
                '--device', 'GPU', '--maxiter', '8', '--layers', '1', '--shots', '4096',
                '--output', 'six_gpu_result.json'], cwd=root, check=True)


## Inspect and download the actual result

An impossible environment, an optimizer that did not converge, and zero feasible
QAOA samples are different outcomes. The exact optimum comes from 120 classical
tours and is never substituted for the QAOA answer. Changing playback speed or
sampling shots does not change the qubit count. No quantum speedup is claimed.


In [ ]:
import json
from pathlib import Path
from google.colab import files
result_path = Path.cwd() / 'qroute_gpu_source/six_gpu_result.json'
data = json.loads(result_path.read_text())
assert data['qubits'] == 25 and data['simulator_device'] == 'GPU'
for key in ['qubits', 'iterations', 'elapsed_seconds', 'feasible_samples',
            'best_feasible_route', 'best_feasible_cost', 'success', 'classical_optimum']:
    print(key, ':', data[key])
files.download(str(result_path))
